# QWEN3-EMBEDDING-0.6B

In [1]:
%pip install --upgrade huggingface-hub

Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [2]:
# %pip install faiss-gpu
%pip install faiss-cpu

   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   - -------------------------------------- 0.8/18.2 MB 8.3 MB/s eta 0:00:03
   ----------- ---------------------------- 5.2/18.2 MB 17.7 MB/s eta 0:00:01
   --------------------- ------------------ 10.0/18.2 MB 20.0 MB/s eta 0:00:01
   -------------------------------- ------- 14.9/18.2 MB 21.8 MB/s eta 0:00:01
   ---------------------------------------- 18.2/18.2 MB 22.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [25]:
%pip install chromadb

  Using cached uvicorn-0.35.0-py3-none-any.whl.metadata (6.5 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached jsonschema-4.25.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached referencing-0.36.2-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.27.1-cp311-cp311-win_amd64.whl.metadata (4.3 kB)
  Using cached websocket_client-1.8.0-py3-none-any.whl.metadata (8.0 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached watchfiles-1.1.0-cp311-cp311-win_amd64.whl.metadata (5.0 kB)
   ---------------------------------------- 0

In [ ]:

# git bash (Docker) ======  GPU current version  ===================

# docker run --gpus all -p 8080:80 -v hf_cache:/data --pull always \
# ghcr.io/huggingface/text-embeddings-inference:1.8 \
# --model-id Qwen/Qwen3-Embedding-0.6B \
# --dtype float16

# ===============================================================


In [18]:
from toolhelper import *

In [19]:
docs = run_load_data_to_embedding('../hoanghamobile.csv')
docs = run_normalization_data(docs, path_stopwords='../stopwords-vietnamese.txt')

In [21]:
docs[:5]

['666baeb49793e149fe7393b4, https://hoanghamobile.com/dien-thoai/nokia-3210-4g-chinh-hang, nokia 3210 4g - chính hãng, , Công nghệ màn hình IPS Kích thước màn hình 24 inch Độ phân giải 2MP Hệ điều hành S30 Bộ nhớ trong 128MB3 RAM 64MB Mạng di động 2G, 3G, 4G, Hỗ trợ VoLTE2 Số khe SIM Hai SIM Nano SIM Nano SIM Dung lượng pin 1450 mAh, 1590000 vnd, Màu Vàng , Xanh , Màu Đen',
 '666baeb49793e149fe7393bc, https://hoanghamobile.com/dien-thoai-di-dong/samsung-galaxy-a05s-6gb-128gb-bh%C4%91t, samsung galaxy a05s - 6gb/128gb (bhđt), Ưu đãi trả góp 0 qua Shinhan Finance hoặc Mirae Asset Finance Giảm 5 không giới hạn khuyến mãi qua Homepaylater Giảm thêm tới 700000 đ khi thanh toán qua Kredivo Giảm 50 tối đa 700 k khi mở thẻ tín dụng Vpbank trên SenID Giảm 20 tối đa 500 k khi mở thẻ tín dụng TPBank EVO Mở thẻ tín dụng VIB Nhận Voucher 600000 đ Giảm 1 tối đa 100000 đ khi thanh toán qua Zalopay, Công nghệ màn hình PLS LCD, 90H z Độ phân giải FHD 2400 x 1080 , 50MP F1 8 AF, 2MP F2 4 , 2MP F2 4 , 13

In [ ]:
# TO UPLOAD DOCUMENT + DATABASE sqLite3


from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.vectorstores import Chroma

# Kết nối tới embedding server
# Khai báo embeddings với batch_size=32
embeddings = HuggingFaceEndpointEmbeddings(
    model="http://localhost:8080"
)
texts = docs
# Danh sách documents (vd: sản phẩm)

embeddings_docs = []
for row in docs:
    embeddings_docs.append(Document(page_content=row))

text_vectors = []
# 3. Tạo embedding vectors cho mỗi text
for i in range(0, len(texts), 32):
    docs_32 = embeddings.embed_documents(texts[i:i + 32])

    for j in docs_32:
        text_vectors.append(j)
# # 4. Ghép text + vector thành pairs
text_embedding_pairs = list(zip(texts, text_vectors))

print(len(texts), len(text_embedding_pairs))  # 319 319

# # # 5. Tạo FAISS index từ embeddings có sẵn
vectorstore = FAISS.from_embeddings(
    text_embeddings=text_embedding_pairs,
    embedding=embeddings
)

vectorstore.save_local('data')
    
vectorstore = Chroma.from_texts(texts[i:i+32], embeddings, persist_directory="./chroma_db")
# Persist (commit xuống DB)
vectorstore.persist()

319 319


# DON'T EMBEDDING DOCUMENT

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
from langchain.vectorstores import FAISS

# Kết nối tới embedding server
# Khai báo embeddings với batch_size=32
embeddings = HuggingFaceEndpointEmbeddings(
    model="http://localhost:8080"
)

vectorstore = FAISS.load_local(
    "./data",
    embeddings,
    allow_dangerous_deserialization=True
)

results = vectorstore.similarity_search("điện thoại iphone nào tốt nhất", k=8)
for r in results:
    print(r.page_content)

666baeb99793e149fe7394ec, https://hoanghamobile.com/dien-thoai-di-dong/iphone-11-128gb-chinh-hang-vn-a, điện thoại iphone 11 (128gb) - chính hãng vn/a, KM 1 Ưu đãi trả góp 0 qua thẻ tín dụng, Công nghệ màn hình IPS LCD Độ phân giải 828 x 1792 Pixels, 2 camera 12 MP, 12 MP Kích thước màn hình 61 inch Hệ điều hành iOS 15 Vi xử lý Apple A13 Bionic 6 nhân Bộ nhớ trong 128 GB RAM 4GB Mạng di động Hỗ trợ 4G Số khe SIM 1 Nano SIM 1 eSIM Dung lượng pin 3110 mAh, 9790000 vnd, White , Black , Red , Green , Purple
666baeb89793e149fe7394a8, https://hoanghamobile.com/dien-thoai-di-dong/apple-iphone-13-128gb-chinh-hang-vn-a, điện thoại iphone 13 (128gb) - chính hãng vn/a, KM 1 Ưu đãi trả góp 0 qua thẻ tín dụng, Công nghệ màn hình OLED Độ phân giải 1170 x 2532 Pixels, 2 camera 12 MP, 12 MP Hệ điều hành iOS 15 Vi xử lý Apple A15 Bộ nhớ trong 128 GB RAM 4GB Mạng di động Hỗ trợ 5G Số khe SIM 1 Nano SIM 1 eSIM, 13190000 vnd, Green , Red , Midnight , Pink , Blue , Starlight
666baeb89793e149fe739476, https

In [56]:
vectorstore = FAISS.load_local(
    folder_path="./data",
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
    )
query = "có cái nào snapdragon không nhỉ?"
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 320})
results = retriever.get_relevant_documents(query)
for r in results:
    print(r.page_content)

666baeb69793e149fe73941a, https://hoanghamobile.com/dien-thoai-di-dong/nothing-phone-2, điện thoại nothing phone 2, Giảm 5 không giới hạn khuyến mãi qua Homepaylater Giảm 50 tối đa 700 k khi mở thẻ tín dụng Vpbank trên SenID Giảm 20 tối đa 500 k khi mở thẻ tín dụng TPBank EVO, , ,
666baeb89793e149fe73947c, https://hoanghamobile.com/dien-thoai-di-dong/nokia-8210-4g-chinh-hang, điện thoại nokia 8210 4g - chính hãng, Giảm 5 không giới hạn khuyến mãi qua Homepaylater Giảm 50 tối đa 700 k khi mở thẻ tín dụng Vpbank trên SenID Giảm 20 tối đa 500 k khi mở thẻ tín dụng TPBank EVO Giảm 1 tối đa 100000 đ khi thanh toán qua Zalopay, Công nghệ màn hình Đang cập nhật Độ phân giải QVGA 240 x 320 Pixels , 03 MP Hệ điều hành Series 30 Vi xử lý Unisoc T107 Bộ nhớ trong 128 MB RAM 48MB Mạng di động Hỗ trợ 4G Số khe SIM 2 Nano SIM Dung lượng pin 1450 mAh, 1490000 vnd, Màu Vàng , Xanh , Đỏ
666baeb89793e149fe739491, https://hoanghamobile.com/dien-thoai-di-dong/itel-9210-4g-chinh-hang, điện thoại itel 9210 

# ===================== NEW =====================

In [1]:
%pip install -U textembed

  Using cached textembed-0.0.8-py3-none-any.whl.metadata (6.1 kB)
  Using cached annotated_types-0.6.0-py3-none-any.whl.metadata (12 kB)
  Using cached anyio-4.3.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached certifi-2024.2.2-py3-none-any.whl.metadata (2.2 kB)
  Using cached charset_normalizer-3.3.2-cp311-cp311-win_amd64.whl.metadata (34 kB)
  Using cached click-8.1.7-py3-none-any.whl.metadata (3.0 kB)
  Using cached dnspython-2.6.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached email_validator-2.1.1-py3-none-any.whl.metadata (26 kB)
  Using cached fastapi-0.111.0-py3-none-any.whl.metadata (25 kB)
  Using cached fastapi_cli-0.0.3-py3-none-any.whl.metadata (7.0 kB)
  Using cached filelock-3.14.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached fsspec-2024.5.0-py3-none-any.whl.metadata (11 kB)
  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached httpcore-1.0.5-py3-none-any.whl.metadata (20 kB)
  Using cached httptools-0.6.1-cp311-cp311-win_amd64.whl.metada

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [18 lines of output]
      Traceback (most recent call last):
        File "d:\langchain\lang\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
        File "d:\langchain\lang\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "d:\langchain\lang\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
                 ^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\lea26\AppData\Local\Temp\pip-build-env-_1_d1qlc\overlay\Lib\site-packages\setuptools\build_meta.py", line 331, in get_requires_for_build_wheel
          return

In [ ]:
%python -m textembed.server --models sentence-transformers/all-MiniLM-L12-v2 --workers 4 --api-key TextEmbed 

In [1]:
from langchain_community.embeddings import TextEmbedEmbeddings

In [ ]:
embeddings = TextEmbedEmbeddings(
    model="sentence-transformers/all-MiniLM-L12-v2",
    api_url="http://0.0.0.0:8000/v1",
    api_key="TextEmbed",
)

In [ ]:
# Define a list of documents
documents = [
    "Data science involves extracting insights from data.",
    "Artificial intelligence is transforming various industries.",
    "Cloud computing provides scalable computing resources over the internet.",
    "Big data analytics helps in understanding large datasets.",
    "India has a diverse cultural heritage.",
]

# Define a query
query = "What is the cultural heritage of India?"

In [ ]:
# Embed all documents
document_embeddings = embeddings.embed_documents(documents)

# Embed the query
query_embedding = embeddings.embed_query(query)

In [ ]:
# Compute Similarity
import numpy as np

scores = np.array(document_embeddings) @ np.array(query_embedding).T
dict(zip(documents, scores))

# ===================== NEW =====================

In [ ]:
%pip install transformers torch

In [ ]:
%pip install llama-cpp-python

In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "qwen/qwen3-embedding-0.6b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at qwen/qwen3-embedding-0.6b and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Qwen3ForSequenceClassification(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151669, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_l